# 12 · Capstone: from warm-up notebook to tested package

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Day 2 | about 80 minutes live, followed by Show & Tell | [11 · From notebook to package](11_from_notebook_to_package.ipynb) | [X1 · Git, DVC and MLflow](X1_git_dvc_mlflow.ipynb) |

On the morning of day 1 you looked at a churn notebook that worked but would hurt anyone who had to reuse,
test, schedule or trust it. Its worst problem was invisible: next month's customers were encoded differently
depending on who else was in the file, so predictions changed silently.

Now you rebuild it as a small Python package, `churn_capstone`, using what you learned over the two days.
You are not starting from a blank page: a set of **acceptance tests** in `tests_capstone/` describes what the
package must do. Your job is to make them pass, one at a time, with code you would be happy to show a colleague.

> **Learning goals.** By the end you can
>
> - turn notebook code into modules with clear responsibilities
> - write classes, exceptions and a config object that satisfy a given interface
> - use failing tests as a to-do list and run them from the terminal
> - explain one design decision of your package and its trade-offs to others
> - compare your design with a reference implementation and judge the differences

**Before you start.** You should have worked through notebooks 01 to 11. Open a terminal in the repository
root. Work in pairs.

The tests come in three levels: **Core** (everyone), **Stretch** (if you finish early), **Challenge** (for
experienced programmers). Getting all Core tests green is a real achievement. Nobody is expected to finish
everything in 80 minutes.

**In this notebook**

1. How to work
2. The contract
3. Suggested order of work
4. Try your package
5. If you finish early: extension menu
6. Compare with the reference implementation
7. Show & Tell and retrospective

---

## 1. How to work

**Pair programming.** One person types (the *driver*), the other reads, thinks ahead and checks against the
contract (the *navigator*). Swap roles every 15 minutes, for example at every green test. If the two of you
have different experience levels, let the less experienced person drive first: the navigator explains, the
driver decides.

**The loop.**

1. Run the tests of your current level and pick the first failing one.
2. Read the test (it is short) to see exactly what is expected.
3. Write the smallest amount of clean code that makes it pass.
4. Run the tests again. Green? Commit to memory what you did, then tidy up (names, docstrings, type hints)
   while the tests protect you.

**Commands** (terminal, repository root):

```bash
uv run pytest tests_capstone/test_1_core.py          # Core level
uv run pytest tests_capstone/test_1_core.py -x       # stop at the first failure
uv run pytest tests_capstone/test_1_core.py -k load  # only tests with "load" in the name
uv run pytest tests_capstone                         # everything
uv run ruff check src/churn_capstone                 # lint your package
```

Your code goes into `src/churn_capstone/`, next to the existing `__init__.py`. The package is already installed
(notebook 11), so the tests and this notebook can import it as soon as you create a file.

You can also run the tests from this notebook with the helper below. It prints pytest's summary.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("..").resolve()
DATA_PATH = REPO / "data" / "raw" / "telco_churn.csv"


def run_capstone_tests(level: str = "test_1_core.py", last_lines: int = 25) -> None:
    """Run one level of the acceptance tests and show the end of pytest's report."""
    run = subprocess.run(
        [sys.executable, "-m", "pytest", f"tests_capstone/{level}", "--tb=no", "-rfE"],
        cwd=REPO,
        capture_output=True,
        text=True,
        env={**os.environ, "PY_COLORS": "0", "COLUMNS": "160"},  # plain, untruncated report
    )
    lines = run.stdout.strip().splitlines()
    print("\n".join(lines[-last_lines:]))


run_capstone_tests("test_1_core.py")

Before you write any code, every test fails with a message like *"Create the module
src/churn_capstone/data.py"*. That is your to-do list.

---

## 2. The contract

These are the names the tests import. Spell them exactly like this; everything else (helper functions,
internal classes, how you split the work inside a module) is your design decision.

**Core** (`tests_capstone/test_1_core.py`)

| Where | What | Behaviour the tests check |
|---|---|---|
| `churn_capstone/data.py` | `load_customers(path)` | Returns a DataFrame with **all 7043** customers; `TotalCharges` is a float column without gaps; `Churn` is `0`/`1`; `customerID` is not a column (make it the index or drop it). |
| `churn_capstone/errors.py` | `SchemaError` | `load_customers` raises it when a required column (for example `tenure`) is missing from the file. |
| `churn_capstone/model.py` | `ChurnModel()` | Can be created without arguments. `fit(X, y)` returns the model itself. `predict(X)` returns one 0/1 label per row. |
| | | **A customer's prediction must not depend on who else is in the batch** (the warm-up bug). |

Here `X` is the DataFrame from `load_customers` without the `Churn` column, and `y` is the `Churn` column.
`X` contains every other column of the file: your model chooses which ones to use.

**Stretch** (`tests_capstone/test_2_stretch.py`)

| Where | What | Behaviour the tests check |
|---|---|---|
| `churn_capstone/config.py` | `ExperimentConfig` | Can be created without arguments; has `test_size` (between 0 and 1) and `random_state` (an int). `ExperimentConfig(test_size=1.5)` raises a `ValueError` when the object is **created**. Same for 0, 1 and negative values. |
| `churn_capstone/errors.py` | `ChurnError`, `NotFittedError` | `SchemaError` and `NotFittedError` are subclasses of `ChurnError`. |
| `churn_capstone/model.py` | `ChurnModel(config)` | Accepts a config as the first argument. `predict` before `fit` raises `NotFittedError`. |
| | | Useful model: F1 above 0.5 on a holdout set of unseen customers (predicting "nobody churns" scores 0). |

**Challenge** (`tests_capstone/test_3_challenge.py`)

| Where | What | Behaviour the tests check |
|---|---|---|
| `churn_capstone/errors.py` | `DataQualityError` | Raised by `load_customers` when `TotalCharges` is blank for a customer whose `tenure` is above 0. |
| `churn_capstone/model.py` | `ChurnModel.predict` | A category never seen in training (a new `Contract` value `"Three year"`) does not crash prediction. |
| | `model.save(path)`, `ChurnModel.load(path)` | Saving and loading again gives identical predictions. |

Each test file starts with a docstring repeating its part of the contract. When in doubt, the test code is the
specification.

---

## 3. Suggested order of work

The steps follow the tests. Each step names the notebook where the idea was introduced. Hints are folded; open
them only when you are stuck for more than a few minutes.

### Step 1 · Core: an exception for missing columns (notebook 09)

Create `src/churn_capstone/errors.py` with a `SchemaError`. Make it useful: store the list of missing columns
as an attribute and put it into the message.

<details><summary><b>Show hint</b></summary>

A custom exception is a class that inherits from `Exception` (or, at Stretch level, from your own base class).
Call `super().__init__(message)` so the message shows up in tracebacks.

```python
class SchemaError(Exception):
    def __init__(self, missing_columns: list[str]) -> None:
        self.missing_columns = missing_columns
        super().__init__(f"Missing required columns: {missing_columns}")
```

</details>

### Step 2 · Core: loading and cleaning the data (notebooks 00, 07, 09)

Create `src/churn_capstone/data.py` with `load_customers(path)`. Look back at the warm-up notebook: which
cleaning steps were right, and which were wrong?

- Check the required columns first and raise `SchemaError` with the missing ones.
- `TotalCharges` is stored as text. Eleven rows are blank. Who are these customers (look at their `tenure`)?
  What is the *correct* value, rather than dropping them? The test expects all 7043 customers.
- Turn `Churn` from `"Yes"`/`"No"` into 1/0.

<details><summary>Hint: what are the blank TotalCharges?</summary>

All 11 blank rows have `tenure == 0`: customers who joined this month and have not been billed yet. Their total
charges are 0, not unknown. Dropping them (as the warm-up did) silently removes exactly the newest customers.

</details>

<details><summary><b>Hint: compact walkthrough</b></summary>

```python
REQUIRED_COLUMNS = ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]

def load_customers(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise SchemaError(missing)
    df = df.set_index("customerID")
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0.0)
    ...  # Churn to 0/1
    return df
```

`errors="coerce"` turns the blanks into NaN. At Challenge level you must check that those NaNs only occur for
`tenure == 0` before filling them.

</details>

### Step 3 · Core: a model that remembers its encoding (notebooks 01, 04, 08)

Create `src/churn_capstone/model.py` with a `ChurnModel` class. This is where you fix the warm-up's hidden bug.

Design questions to settle with your partner before typing:

- Which columns does the model use? Start small: the numeric columns plus `Contract` already give a useful
  model. Add more later.
- What must the model **learn in `fit`** and **reuse in `predict`**, so that the encoding of a customer never
  depends on the rest of the batch?
- Which estimator does it hold (composition, notebook 08)? `LogisticRegression` is a good start.

<details><summary>Hint: what to remember</summary>

The problem in the warm-up was that encoding was fitted on whatever batch came in. If you one-hot encode with
`pd.get_dummies`, a batch containing only "Two year" customers produces only a "Two year" column. Remember the
list of encoded columns at `fit` time and, in `predict`, force every batch into exactly those columns with
`DataFrame.reindex(columns=..., fill_value=0)`. Alternatives: your `OneHotEncoder` from notebook 04, or
scikit-learn's `OneHotEncoder(handle_unknown="ignore")` inside a `ColumnTransformer`.

</details>

<details><summary><b>Hint: compact walkthrough</b></summary>

```python
NUMERIC = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
CATEGORICAL = ["Contract", "InternetService", "PaymentMethod"]

class ChurnModel:
    def __init__(self) -> None:
        self._columns: list[str] = []          # learned in fit
        self._estimator = make_pipeline(StandardScaler(), LogisticRegression())

    def _encode(self, X: pd.DataFrame) -> pd.DataFrame:
        encoded = pd.get_dummies(X[NUMERIC + CATEGORICAL], columns=CATEGORICAL, dtype=int)
        if self._columns:                        # after fit: same columns for every batch
            encoded = encoded.reindex(columns=self._columns, fill_value=0)
        return encoded

    def fit(self, X, y):
        ...  # reset self._columns, encode, remember encoded.columns, fit the estimator
        return self

    def predict(self, X):
        ...
```

Scaling the numeric columns (here with scikit-learn's `StandardScaler`) avoids a `ConvergenceWarning` from
`LogisticRegression`, because `TotalCharges` is in the thousands while `SeniorCitizen` is 0 or 1.

</details>

When all Core tests pass: congratulations, the warm-up bug cannot come back without a test failing. Take two
minutes to add docstrings and type hints, then run `uv run ruff check src/churn_capstone`.

### Step 4 · Stretch: config, error family, useful model (notebooks 07, 09)

- `config.py`: a dataclass `ExperimentConfig` with `test_size` and `random_state` (add more fields if your model
  needs them, for example the feature lists or a threshold). Validate in `__post_init__`; `frozen=True` keeps it
  from being changed by accident.
- `errors.py`: introduce `ChurnError` as the base class; `SchemaError` and the new `NotFittedError` inherit from
  it.
- `model.py`: `ChurnModel(config=None)`; raise `NotFittedError` from `predict` if `fit` was not called.
- If the "useful model" test fails, check which features you use. Which column had the strongest relationship
  with churn in the warm-up?

<details><summary><b>Show hint</b></summary>

```python
@dataclass(frozen=True)
class ExperimentConfig:
    test_size: float = 0.2
    random_state: int = 42

    def __post_init__(self) -> None:
        if not 0 < self.test_size < 1:
            raise ValueError(f"test_size must be between 0 and 1, got {self.test_size}")
```

In the model: `self.config = config or ExperimentConfig()`, and use `self.config.random_state` for the
estimator. For "not fitted", an empty `self._columns` list (or an explicit `self._fitted` flag) tells you
whether `fit` ran.

</details>

### Step 5 · Challenge: robustness and persistence (notebooks 04, 09, 10)

- `DataQualityError`: blanks in `TotalCharges` are only acceptable for `tenure == 0`. Anything else means the
  export is broken, and the error message should say which customers are affected.
- Unseen categories: check what your encoding does with `Contract == "Three year"`. Should it crash, warn, or
  silently encode as zeros? Decide, and log a warning if you encode silently.
- `save(path)` and the class method `load(path)`: `joblib.dump(self, path)` and `joblib.load(path)` are enough.
  Should `load` check that the file really contains a `ChurnModel`?

<details><summary><b>Show hint</b></summary>

```python
total = pd.to_numeric(df["TotalCharges"], errors="coerce")
broken = total.isna() & (df["tenure"] > 0)
if broken.any():
    raise DataQualityError(f"TotalCharges missing for customers {list(df.index[broken][:5])}")
```

`@classmethod def load(cls, path)` returns `joblib.load(path)`; `ChurnModel.load(path)` then works without an
instance.

</details>

---

## 4. Try your package

Tests tell you whether the contract holds. It is also worth *using* your package the way a colleague would.
The cells below work before and after you have written your modules.

In [ ]:
try:
    from churn_capstone.data import load_customers

    customers = load_customers(DATA_PATH)
    print(customers.shape)
    print(customers[["tenure", "TotalCharges", "Contract", "Churn"]].head())
except ImportError as exc:
    print(f"Not there yet: {exc}")

In [ ]:
import pandas as pd

try:
    from churn_capstone.data import load_customers
    from churn_capstone.model import ChurnModel

    customers = load_customers(DATA_PATH)
    X, y = customers.drop(columns=["Churn"]), customers["Churn"]
    model = ChurnModel().fit(X, y)

    # Same five customers, scored alone and as part of the full table
    five = X[X["Contract"] == "Month-to-month"].head(5)
    in_full_batch = pd.Series([int(v) for v in model.predict(X)], index=X.index)
    print("Scored alone:          ", [int(v) for v in model.predict(five)])
    print("Scored in a full batch:", in_full_batch.loc[five.index].tolist())
    print(model)
except ImportError as exc:
    print(f"Not there yet: {exc}")

Notice what `print(model)` shows. If it is something like `<churn_capstone.model.ChurnModel object at
0x7f...>`, that is a hint for a small improvement (notebook 03).

---

## 5. If you finish early: extension menu

Pick one. Each is a realistic next step for a team that owns this model.

| Extension | What to do | Where to look |
|---|---|---|
| Command line | `__main__.py` with `argparse`: train from a config file, print metrics, save the model; configure logging there. Register it in `[project.scripts]`. | `src/churn/__main__.py`, notebook 11 |
| Business threshold | Pick the decision threshold that minimises the cost of a retention campaign instead of using 0.5. Use a validation split, not the test set. | `RetentionCost` in notebook 03, `src/churn/evaluation.py` |
| Experiment tracking | Log config, metrics and the saved model of each run with MLflow. | Notebook X1 |
| Your own tests | Add unit tests for your cleaning rules in `playground/` or a new `tests_capstone/test_4_mine.py`. | Notebook 10 |
| scikit-learn compatibility | Turn your encoder into a real scikit-learn transformer and use `GridSearchCV`. | Notebook X2 |
| Serving (design only) | Sketch how a FastAPI endpoint would score one customer with the saved model. Which layer does it belong to? What must happen at startup, what per request? | Notebook 08 |

---

## 6. Compare with the reference implementation

`src/churn` solves the same problem. Open it **after** you have tried yourself, and discuss with your partner:

1. How does `src/churn` make sure the encoding never depends on the batch? Where does that knowledge live?
2. `churn.model.ChurnModel` receives an `estimator` argument. What does that make possible, and which test in
   `tests/test_model.py` uses it?
3. `churn.data` has a `CustomerLoader` Protocol and two loaders. Did you need that flexibility? When would you?
4. Name one thing your design does better or more simply than the reference. Simpler is often better.

<details><summary>Notes for the discussion</summary>

1. In `OneHotEncoder` (`src/churn/features.py`): categories are learned in `_fit` and replayed in `_transform`.
   `ChurnModel` composes it inside a `FeaturePipeline`.
2. Any object with `fit` and `predict_proba` can be plugged in (composition plus a Protocol). The test
   `test_any_probabilistic_estimator_can_be_plugged_in` uses scikit-learn's `DummyClassifier`.
3. Loaders make the data source replaceable (CSV today, a database tomorrow) and make end-to-end tests possible
   without files (`InMemoryLoader`, see `tests/test_experiment.py`). For a one-off analysis it would be
   over-engineering; for a model that runs every month it pays off quickly.
4. Typical answers: fewer classes, `pd.get_dummies` plus `reindex` instead of a custom encoder, no Protocols.
   All fine if the tests pass and the code is easy to read.

</details>

---

## 7. Show & Tell and retrospective

**Show & Tell (2 minutes per pair).** Show your package in the editor, not slides:

- one design decision you are proud of,
- one decision you are unsure about, and the alternative you considered.

Listeners: ask one question per pair, starting with "Why did you...".

**Retrospective (individually, then share).** Write one sentence each:

- *Start*: one practice from this workshop you will start using next week.
- *Stop*: one habit from your current notebooks you want to stop.
- *Continue*: one thing you already did well and will keep doing.

**Transfer task (after the workshop).** Take one notebook from your own work. Identify the piece of code you
copy most often between notebooks, move it into a small package with one test, and import it from the
notebook. That is the whole workshop in one step.

---

## Summary

- Acceptance tests turn a vague goal ("make it production-ready") into a concrete to-do list.
- The warm-up bug is fixed by an object that **learns** its encoding in `fit` and **replays** it in `predict`,
  and a test now guards that behaviour.
- Small modules with one job each (`errors`, `data`, `config`, `model`) are easier to read, test and change than
  one long notebook.
- Exceptions with meaningful names and messages make failures easy to diagnose; a config dataclass catches bad
  settings at creation time.
- There is more than one good design. Explain your trade-offs, and keep it simple until the problem demands more.

---

## Sources and further reading

**Sources.** This notebook is based on:

- [Martin Fowler, *Refactoring*, 2nd edition](https://martinfowler.com/books/refactoring.html): the definition we use, changing structure without changing behaviour, in small, tested steps.
- [Birgitta Böckeler and Nina Siessegger, *On Pair Programming*](https://martinfowler.com/articles/on-pair-programming.html): driver and navigator, and why rotating helps.

**For self-study.** Where to go next if you want to learn more:

- [Refactoring Guru: Refactoring](https://refactoring.guru/refactoring) (website, beginner): code smells and the refactoring that fixes each.
- [Google Engineering Practices: Code review](https://google.github.io/eng-practices/review/) (guide, beginner to intermediate): what reviewers look for, useful before you show your package to colleagues.
- [Harry Percival and Bob Gregory, *Architecture Patterns with Python*](https://www.cosmicpython.com/) (O'Reilly 2020, free to read online) (book, intermediate to advanced): where to go after the capstone.
- [FastAPI tutorial](https://fastapi.tiangolo.com/tutorial/) (docs, intermediate), for the "serve the model" extension.

---

[← 11 · From notebook to package](11_from_notebook_to_package.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [X1 · Git, DVC and MLflow →](X1_git_dvc_mlflow.ipynb)